# Haystack Agents with Persistent Memory

Build Haystack agents that remember user preferences and past interactions across conversations using Hindsight memory.

This notebook demonstrates:
- Creating Hindsight memory tools for a Haystack `Agent`
- Storing information with `retain_memory`
- Retrieving memories across sessions with `recall_memory`
- Synthesizing knowledge with `reflect_on_memory`
- Automatic recall and retain with `HindsightToolset` — no explicit tool calls needed

## Prerequisites

- Python 3.10+
- A [Hindsight Cloud](https://ui.hindsight.vectorize.io/signup) account **or** a self-hosted Hindsight instance
- An OpenAI API key (or any Haystack-supported chat model)

### Option A: Hindsight Cloud

Sign up at [ui.hindsight.vectorize.io](https://ui.hindsight.vectorize.io/signup) and grab your API key from the dashboard.

### Option B: Self-Hosted (Docker)

```bash
export OPENAI_API_KEY=your-key

docker run --rm -it --pull always -p 8888:8888 -p 9999:9999 \
  -e HINDSIGHT_API_LLM_API_KEY=$OPENAI_API_KEY \
  -e HINDSIGHT_API_LLM_MODEL=gpt-4o-mini \
  -v $HOME/.hindsight-docker:/home/hindsight/.pg0 \
  ghcr.io/vectorize-io/hindsight:latest
```

## Installation

In [ ]:
!pip install hindsight-haystack haystack-ai openai python-dotenv -U

## Setup

Configure the Hindsight client. Set `HINDSIGHT_API_URL` and `HINDSIGHT_API_KEY` in your `.env` file or environment:

| | Hindsight Cloud | Self-Hosted |
|---|---|---|
| `HINDSIGHT_API_URL` | `https://api.hindsight.vectorize.io` | `http://localhost:8888` |
| `HINDSIGHT_API_KEY` | Your cloud API key | *(not required for local)* |

> **Note:** You also need `OPENAI_API_KEY` set in your environment or `.env` file for the Haystack chat model.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

HINDSIGHT_API_URL = os.getenv("HINDSIGHT_API_URL", "http://localhost:8888")
HINDSIGHT_API_KEY = os.getenv("HINDSIGHT_API_KEY")  # Required for Hindsight Cloud

from hindsight_client import Hindsight

client_kwargs = {"base_url": HINDSIGHT_API_URL}
if HINDSIGHT_API_KEY:
    client_kwargs["api_key"] = HINDSIGHT_API_KEY

client = Hindsight(**client_kwargs)

## Create Memory Tools

`create_hindsight_tools()` returns Haystack `Tool` objects that the agent can call.
The `mission` parameter auto-creates the memory bank on first use.

In [ ]:
# Clean up any leftover bank from a previous run
from hindsight_haystack.tools import _run_sync

try:
    _run_sync(client.adelete_bank("demo-haystack"))
except Exception:
    pass

In [ ]:
from hindsight_haystack import create_hindsight_tools

tools = create_hindsight_tools(
    client=client,
    bank_id="demo-haystack",
    mission="Track user preferences, background, and project context",
    tags=["source:chat"],
    retain_context="haystack-cookbook",
)

print(f"Tools created: {[t.name for t in tools]}")

## Build the Agent

Create a Haystack `Agent` with the memory tools and an OpenAI chat generator.

In [ ]:
from haystack.components.agents import Agent
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.dataclasses import ChatMessage

agent = Agent(
    chat_generator=OpenAIChatGenerator(model="gpt-4o-mini"),
    tools=tools,
    system_prompt=(
        "You are a helpful assistant with long-term memory. "
        "Use retain_memory to store important facts about the user. "
        "Use recall_memory to search your memory before answering. "
        "Use reflect_on_memory for thoughtful summaries of what you know."
    ),
)

## Conversation 1: Store Preferences

Tell the agent about yourself. It will use `retain_memory` to store the information in Hindsight.

In [ ]:
result = agent.run(
    messages=[
        ChatMessage.from_user(
            "Hi! I'm Alice. I'm a data scientist who works with Python and SQL. "
            "I prefer dark mode and use VS Code. I'm currently working on a "
            "recommendation engine for an e-commerce platform."
        )
    ]
)
print(f"\nAgent: {result['messages'][-1].text}")

In [ ]:
import time

# Hindsight processes retained content asynchronously (extracting facts, entities, embeddings).
# The sleep gives the server time to finish before we recall.
time.sleep(3)

## Conversation 2: Recall Across Sessions

Create a new agent instance — simulating a new session. Memory persists because it's
stored in Hindsight, not in the agent. The agent uses `recall_memory` to find relevant facts.

In [ ]:
# New agent instance — fresh session, same memory bank
agent2 = Agent(
    chat_generator=OpenAIChatGenerator(model="gpt-4o-mini"),
    tools=tools,
    system_prompt=(
        "You are a helpful assistant with long-term memory. "
        "Use recall_memory to search your memory before answering questions."
    ),
)

result = agent2.run(
    messages=[ChatMessage.from_user("What IDE do I use? And what's my current project?")]
)
print(f"\nAgent: {result['messages'][-1].text}")

## Reflect: Synthesize Knowledge

Use `reflect_on_memory` to get a synthesized, reasoned answer that draws on the full knowledge graph — not just raw facts.

In [ ]:
result = agent2.run(
    messages=[
        ChatMessage.from_user(
            "Based on everything you know about me, what tools and libraries "
            "would you recommend for my current project?"
        )
    ]
)
print(f"\nAgent: {result['messages'][-1].text}")

## Selective Tools

Use `include_retain`, `include_recall`, and `include_reflect` to control which tools are exposed.

In [ ]:
# Create only retain + recall tools (no reflect)
selective_tools = create_hindsight_tools(
    client=client,
    bank_id="demo-haystack",
    tags=["source:chat"],
    budget="mid",
    include_reflect=False,
)

print(f"Selective tools: {[t.name for t in selective_tools]}")

## Automatic Recall & Retain with HindsightToolset

Instead of relying on the agent to call tools explicitly, `HindsightToolset` can
automatically recall relevant memories into the system prompt before each turn and
retain user/assistant messages after each turn — similar to how the LlamaIndex and
Pydantic AI integrations work.

Use `toolset.run(agent, messages=...)` instead of `agent.run(messages=...)` to
enable this behavior.

In [ ]:
from hindsight_haystack import HindsightToolset

toolset = HindsightToolset(
    client=client,
    bank_id="demo-haystack-auto",
    mission="Track user preferences and context",
    tags=["source:chat"],
    retain_context="haystack-cookbook",
    auto_recall=True,   # Inject memories into system prompt before each turn
    auto_retain=True,    # Store user + assistant messages after each turn
)

auto_agent = Agent(
    chat_generator=OpenAIChatGenerator(model="gpt-4o-mini"),
    tools=toolset,  # Pass the toolset — agent can still use tools explicitly too
    system_prompt="You are a helpful assistant with long-term memory.",
)

# Tell the agent something — auto_retain stores it automatically
result = toolset.run(
    auto_agent,
    messages=[ChatMessage.from_user(
        "I'm a backend engineer who uses Go and PostgreSQL. "
        "I prefer terminal-based workflows and use tmux + neovim."
    )],
)
print(f"\nAgent: {result['messages'][-1].text}")

In [ ]:
time.sleep(3)  # Wait for async processing

# Ask a new question — auto_recall injects relevant memories into the system prompt
# before the agent even starts reasoning. No explicit tool call needed.
result = toolset.run(
    auto_agent,
    messages=[ChatMessage.from_user("What editor do I use?")],
)
print(f"\nAgent: {result['messages'][-1].text}")

## Cleanup

In [ ]:
from hindsight_haystack.tools import _run_sync

_run_sync(client.adelete_bank("demo-haystack"))
_run_sync(client.adelete_bank("demo-haystack-auto"))
print("Banks deleted.")

## Key Takeaways

- **Tool Factory** (`create_hindsight_tools`): Returns Haystack `Tool` objects — pass directly to `Agent(tools=...)`.
- **Three tools**: `retain_memory` (store), `recall_memory` (search), `reflect_on_memory` (synthesize from knowledge graph).
- **Auto-memory** (`HindsightToolset`): Use `auto_recall=True` and `auto_retain=True` with `toolset.run()` for automatic memory without explicit tool calls.
- **Bank missions**: Use `mission=` to auto-create banks with context for fact extraction — no manual `create_bank` step needed.
- **Per-user banks**: Use `bank_id=f"user-{user_id}"` for per-user memory isolation.
- **Tags & context**: Scope memories by source, conversation, or topic for precise recall.